# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² colorectal cancer dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their Croissant `@id` as per best practices for FAIR data.

### Dataset Source
The dataset source is provided via a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nDataset Croissant `@id`:", metadata.id)
print("License:", metadata.license)
print("Version:", metadata.version)
print("Data collection timeframe:", getattr(metadata, 'dataCollectionTimeframe', None))
print("Number of available record sets:", len(metadata.record_sets))

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id` entries. This includes mapping out the data's main table(s) and the unique field identifiers.

In [ ]:
# List all record sets and display their Croissant @id
print("Available record sets and their fields:")

for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}, Name: {rs.name if hasattr(rs,'name') else ''}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, Name: {field.name if hasattr(field,'name') else ''}, DataType: {getattr(field, 'data_type', None)}")
    else:
        print("    (No fields found)")
    print()

# Extract list of all record set @ids for next steps
record_set_ids = [rs.id for rs in metadata.record_sets]

## 3. Data Extraction
Load each record set into a Pandas DataFrame for further exploration. All record set and field references use Croissant `@id` as shown above.

In [ ]:
dataframes = {}
print("Extracting record sets:")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Loaded RecordSet: {record_set_id} | Shape: {df.shape}")

# For demonstration, use the first record set if available
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"\nColumns for record set @id '{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.to_list())
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply several exploratory operations using `@id`-based referencing for fields/columns, such as filtering, normalizing numeric variables, and group-wise summaries.

In [ ]:
# EDA using column @id (as shown in overview; customize IDs as needed)

# Select record set to analyze
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

# For demonstration, attempt to automatically pick a numeric field using datatype, otherwise use fallback by guessing
numeric_field_id = None
for rs in metadata.record_sets:
    if rs.id == record_set_id:
        # Look for fields marked as Numeric
        for field in getattr(rs,'fields', []):
            if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
                if field.id in df.columns:
                    numeric_field_id = field.id
                    break
        if numeric_field_id is None and rs.fields:
            # fallback: just select the first field in DataFrame with int/float dtype
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
        break

if numeric_field_id is None:
    print("No numeric field could be found for EDA.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

    # Filter example
    threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as an interesting cutoff
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: ({len(filtered_df)})")
    display(filtered_df.head(3))

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

    # Try to find a categorical/grouping field for groupby
    group_field_id = None
    for rs in metadata.record_sets:
        if rs.id == record_set_id and hasattr(rs, 'fields'):
            for field in rs.fields:
                if getattr(field, 'data_type', '').lower() in ['text', 'string', 'category'] and field.id in df.columns:
                    if df[field.id].nunique() > 1 and df[field.id].nunique() < len(df):
                        group_field_id = field.id
                        break
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found for groupby in this record set.")

## 5. Visualization
Visualize value distributions or relationships between fields with reference to Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id and there is not too many groups
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load FAIR² colorectal cancer records using the Croissant standard with `mlcroissant` and referenced all entities by Croissant `@id`.
- We overviewed the record set and field structure, loaded tabular data, and performed simple EDA and visualization using numeric and categorical fields (by `@id`).
- For more advanced analysis, users can repeat this process addressing specific record sets or fields as required in their own biomedical research questions.

_For additional details, consult the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) or the [FAIR² dataset specification](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)._